# <font color="lightgreen">ETL Load - Carga de Datos a DuckDB</font>  🦆

	Cargar las tablas del esquema estrella en una base de datos DuckDB para su posterior análisis y consulta.
	Este notebook es **idempotente** y **automatizado**: recrea la base de datos desde los CSVs curados.

---

### <font color="lightgreen">Librerías</font>

In [11]:
import duckdb
import pandas as pd
from pathlib import Path
import logging 
from datetime import datetime


print("Librerías importadas correctamente")
print('Versión de duckdb:', duckdb.__version__)
print('Versión de pandas:', pd.__version__)


Librerías importadas correctamente
Versión de duckdb: 1.5.1
Versión de pandas: 3.0.2


### <font color="lightgreen">Configuración</font>

#### <font color='lightgreen'>Configurar logging<font>

In [12]:
# Configurar logging (consistente con notebooks anteriores)
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

#### <font color='lightgreen'>Configuración de rutas</font>

In [13]:
from pathlib import Path

# Detectamos la raíz del proyecto de forma robusta
current_dir = Path.cwd()

# Caso 1: Estamos dentro de una carpeta llamada 'etl' o 'ETL' (insensible a mayúsculas)
if current_dir.name.lower() == "etl":
    PROJECT_ROOT = current_dir.parent
else:
    # Caso 2: Buscamos un archivo característico de la raíz (README.md o requirements.txt)
    # Subimos niveles hasta encontrar uno de ellos
    PROJECT_ROOT = current_dir
    while not (PROJECT_ROOT / "README.md").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
        PROJECT_ROOT = PROJECT_ROOT.parent
    if not (PROJECT_ROOT / "README.md").exists():
        # Si no encontramos README, asumimos que ya estábamos en la raíz
        PROJECT_ROOT = current_dir

DATA_CURATED = PROJECT_ROOT / "data" / "curated"
DATABASE_DIR = PROJECT_ROOT / "database"
SCHEMA_FILE = DATABASE_DIR / "schema.sql"
DB_FILE = DATABASE_DIR / "technova.duckdb"

# Crear directorio de base de datos si no existe
DATABASE_DIR.mkdir(parents=True, exist_ok=True)

logger.info(f"📁 Raíz del proyecto: {PROJECT_ROOT}")
logger.info(f"📁 Datos curados: {DATA_CURATED}")
logger.info(f"📁 Base de datos: {DB_FILE}")
logger.info(f"📄 Schema: {SCHEMA_FILE}")

2026-04-15 19:37:47,837 - INFO - 📁 Raíz del proyecto: c:\Users\marely\OneDrive\Documentos\Bussiness Inteligence Project\S03-26-Equipo-47-Business-Intelligence-development\S03-26-Equipo-47-Business-Intelligence-development


2026-04-15 19:37:47,840 - INFO - 📁 Datos curados: c:\Users\marely\OneDrive\Documentos\Bussiness Inteligence Project\S03-26-Equipo-47-Business-Intelligence-development\S03-26-Equipo-47-Business-Intelligence-development\data\curated
2026-04-15 19:37:47,843 - INFO - 📁 Base de datos: c:\Users\marely\OneDrive\Documentos\Bussiness Inteligence Project\S03-26-Equipo-47-Business-Intelligence-development\S03-26-Equipo-47-Business-Intelligence-development\database\technova.duckdb
2026-04-15 19:37:47,845 - INFO - 📄 Schema: c:\Users\marely\OneDrive\Documentos\Bussiness Inteligence Project\S03-26-Equipo-47-Business-Intelligence-development\S03-26-Equipo-47-Business-Intelligence-development\database\schema.sql


### <font color="lightgreen">Funciones</font	>

In [14]:
def ejecutar_schema(conn):
    """Ejecuta el archivo schema.sql. Lanza excepción si falla."""
    if not SCHEMA_FILE.exists():
        raise FileNotFoundError(f"Schema no encontrado: {SCHEMA_FILE}")
    
    with open(SCHEMA_FILE, 'r', encoding='utf-8') as f:
        schema_sql = f.read()
    
    conn.execute(schema_sql)
    logger.info("✅ Schema ejecutado correctamente")

In [15]:
def cargar_tabla(conn, tabla, csv_path, columnas):
    """
    Carga una tabla con full-refresh (DELETE + INSERT).
    Valida que el CSV contenga las columnas requeridas.
    Retorna el número de filas cargadas.
    """
    if not csv_path.exists():
        logger.warning(f"  ⚠️ Archivo no encontrado: {csv_path}")
        return 0
    
    df = pd.read_csv(csv_path, encoding='utf-8')
    
    if df.empty:
        logger.warning(f"  ⚠️ Archivo vacío: {csv_path}")
        return 0
    
    # Validar que todas las columnas requeridas existan
    columnas_csv = set(df.columns)
    columnas_requeridas = set(columnas)
    missing = columnas_requeridas - columnas_csv
    if missing:
        raise ValueError(f"Faltan columnas en {csv_path}: {missing}")
    
    # Seleccionar solo las columnas necesarias (por si el CSV tiene extras)
    df = df[columnas]
    
    # DELETE full
    conn.execute(f"DELETE FROM {tabla}")
    
    # INSERT usando registro temporal
    conn.register('temp_df', df)
    columnas_str = ', '.join(columnas)
    conn.execute(f"""
        INSERT INTO {tabla} ({columnas_str})
        SELECT {columnas_str} FROM temp_df
    """)
    
    # Obtener conteo final
    count = conn.execute(f"SELECT COUNT(*) FROM {tabla}").fetchone()[0]
    logger.info(f"  ✅ {tabla}: {count:,} filas cargadas")
    return count

In [16]:
def verificar_fk(conn):
    """Verifica que no haya registros huérfanos en fact_monitoreo y otras tablas."""
    logger.info("\n🔍 Verificando integridad de claves foráneas...")
    
    verificaciones = [
        ('fact_monitoreo', 'id_tiempo', 'dim_tiempo', 'id_tiempo'),
        ('fact_monitoreo', 'id_metrica', 'dim_metrica', 'id_metrica'),
        ('fact_monitoreo', 'id_area', 'dim_area', 'id_area'),
        ('dim_objetivo', 'id_metrica', 'dim_metrica', 'id_metrica')
    ]
    
    todo_ok = True
    for fact_tab, fk_col, dim_tab, pk_col in verificaciones:
        query = f"""
            SELECT COUNT(*) FROM {fact_tab} f
            LEFT JOIN {dim_tab} d ON f.{fk_col} = d.{pk_col}
            WHERE d.{pk_col} IS NULL
        """
        huerfanos = conn.execute(query).fetchone()[0]
        if huerfanos > 0:
            logger.error(f"  ❌ {fact_tab}.{fk_col} -> {dim_tab}: {huerfanos} huérfanos")
            todo_ok = False
        else:
            logger.info(f"  ✅ {fact_tab}.{fk_col} -> {dim_tab}: OK")
    
    return todo_ok

In [17]:
def guardar_metadata(conn, resultados):
    """Registra la carga actual en la tabla metadata_carga."""
    # Crear tabla si no existe (con secuencia para ID autoincremental)
    conn.execute("""
        CREATE TABLE IF NOT EXISTS metadata_carga (
            id INTEGER PRIMARY KEY,
            fecha TIMESTAMP,
            tabla VARCHAR,
            filas INTEGER
        )
    """)
    conn.execute("CREATE SEQUENCE IF NOT EXISTS seq_metadata START 1")
    
    for tabla, filas in resultados.items():
        conn.execute("""
            INSERT INTO metadata_carga (id, fecha, tabla, filas)
            VALUES (nextval('seq_metadata'), ?, ?, ?)
        """, [datetime.now(), tabla, filas])
    
    logger.info("  📝 Metadatos guardados en metadata_carga")

### <font color="lightgreen">Configuración de tablas al cargar</font>

In [18]:
# Definir las tablas con sus archivos CSV y columnas esperadas
# Debe coincidir con lo generado por 03_build_star_schema y con schema.sql
tablas = {
    'dim_tiempo': {
        'csv': DATA_CURATED / 'dim_tiempo.csv',
        'columnas': ['id_tiempo', 'fecha', 'anio', 'mes', 'dia', 
                    'trimestre', 'semana', 'nombre_mes', 'dia_semana', 'es_finde']
    },
    'dim_area': {
        'csv': DATA_CURATED / 'dim_area.csv',
        'columnas': ['id_area', 'nombre_area']
    },
    'dim_metrica': {
        'csv': DATA_CURATED / 'dim_metrica.csv',
        'columnas': ['id_metrica', 'nombre', 'categoria', 'subcategoria', 'unidad', 
                    'formula', 'prioridad', 'frecuencia_recomendada', 
                    'tipo_agregacion', 'tendencia_deseada', 'descripcion']
    },
    'dim_objetivo': {
        'csv': DATA_CURATED / 'dim_objetivo.csv',
        'columnas': ['id_objetivo', 'id_metrica', 'anio', 'valor_objetivo', 
                    'umbral_verde', 'umbral_amarillo', 'peso_relativo']
    },
    'dim_empleado': {
        'csv': DATA_CURATED / 'dim_empleado.csv',
        'columnas': ['id_empleado', 'nombre', 'area', 'genero', 'fecha_ingreso']
    },
    'dim_proveedor': {
        'csv': DATA_CURATED / 'dim_proveedor.csv',
        'columnas': ['id_proveedor', 'nombre_proveedor']
    },
    'dim_categoria': {   # <--- NUEVA tabla, independiente
        'csv': DATA_CURATED / 'dim_categoria.csv',
        'columnas': ['id_categoria', 'nombre_categoria']
    },
    'fact_monitoreo': {
        'csv': DATA_CURATED / 'fact_monitoreo.csv',
        'columnas': ['id_monitoreo', 'id_tiempo', 'id_metrica', 'id_area', 'valor', 'fuente']
    }
}

# Orden de carga: dimensiones primero (sin FK hacia otras), luego hechos
orden_carga = [
    'dim_tiempo', 'dim_area', 'dim_metrica', 'dim_objetivo',
    'dim_empleado', 'dim_proveedor', 'dim_categoria', 
    'fact_monitoreo'
]

### <font color="lightgreen">5. Función principal</font>

In [19]:
def main():
    logger.info("="*60)
    logger.info("🚀 INICIANDO CARGA A DUCKDB")
    logger.info("="*60)
    
    # 🔥 Full-refresh: eliminar base de datos existente
    if DB_FILE.exists():
        DB_FILE.unlink()
        logger.info(f"🗑️ Base de datos existente eliminada: {DB_FILE}")
    
    conn = None
    try:
        # Conectar a DuckDB (crea archivo nuevo)
        conn = duckdb.connect(str(DB_FILE))
        logger.info(f"✅ Conectado a: {DB_FILE}")
        
        # Ejecutar schema.sql (crea tablas vacías)
        ejecutar_schema(conn)
        
        # Cargar tablas en orden
        logger.info("\n📤 EJECUTANDO FULL-REFRESH...")
        resultados = {}
        
        for tabla in orden_carga:
            if tabla not in tablas:
                logger.warning(f"⚠️ Tabla {tabla} no encontrada en configuración")
                continue
            config = tablas[tabla]
            logger.info(f"\n📁 {tabla}")
            filas = cargar_tabla(conn, tabla, config['csv'], config['columnas'])
            resultados[tabla] = filas
        
        # Verificar integridad referencial
        fk_ok = verificar_fk(conn)
        
        # Guardar metadatos de carga
        guardar_metadata(conn, resultados)
        
        # Resumen final
        logger.info("\n" + "="*60)
        logger.info("✅ CARGA COMPLETADA")
        logger.info("="*60)
        logger.info("\n📊 RESUMEN DE FILAS CARGADAS:")
        for tabla, filas in resultados.items():
            logger.info(f"   • {tabla}: {filas:,} filas")
        
        if not fk_ok:
            logger.warning("\n⚠️ ADVERTENCIA: Problemas de integridad referencial detectados")
        
        logger.info(f"\n💾 Base de datos: {DB_FILE}")
        
    except Exception as e:
        logger.error(f"❌ Error crítico: {e}", exc_info=True)
        raise  # Relanzar para que se vea en el notebook
    finally:
        if conn:
            conn.close()
            logger.info("🔒 Conexión cerrada")

### <font color="lightgreen">6. Ejecución</font>

In [20]:
if __name__ == "__main__":
    main()

2026-04-15 19:37:48,162 - INFO - ============================================================
2026-04-15 19:37:48,163 - INFO - 🚀 INICIANDO CARGA A DUCKDB
2026-04-15 19:37:48,165 - INFO - ============================================================
2026-04-15 19:37:48,168 - INFO - 🗑️ Base de datos existente eliminada: c:\Users\marely\OneDrive\Documentos\Bussiness Inteligence Project\S03-26-Equipo-47-Business-Intelligence-development\S03-26-Equipo-47-Business-Intelligence-development\database\technova.duckdb
2026-04-15 19:37:48,339 - INFO - ✅ Conectado a: c:\Users\marely\OneDrive\Documentos\Bussiness Inteligence Project\S03-26-Equipo-47-Business-Intelligence-development\S03-26-Equipo-47-Business-Intelligence-development\database\technova.duckdb
2026-04-15 19:37:48,390 - INFO - ✅ Schema ejecutado correctamente
2026-04-15 19:37:48,392 - INFO - 
📤 EJECUTANDO FULL-REFRESH...
2026-04-15 19:37:48,396 - INFO - 
📁 dim_tiempo
2026-04-15 19:37:48,478 - INFO -   ✅ dim_tiempo: 2,641 filas cargadas
2